# ML-07 — Baseline Action Score and Top-20 Review

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/varshitha922/flyrank-ml-internship/blob/main/work/notebooks/w04_baseline_score.ipynb?flush_cache=true)

This skeleton is yours to fill. Work the sections **in order** — each one has a one-line hint. Simple words, honest numbers.

> Working with an AI assistant? Tell it to read `skills/README.md` first and load the one skill this assignment names on its card.

## 1. My rule and its reason codes

*Write the rule in plain words first. Then the reason codes it can output.*

My rule: Prioritize pages with meaningful search visibility for review. Pages with higher search visibility should receive a higher priority score.

I checked two signals: staleness and search visibility. Staleness was OPPOSITE, so I do not use it in the final rule. Search visibility was CONFIRMED, so the final rule uses search visibility.

Reason codes:

* visible — the page has meaningful search visibility, so it is worth reviewing.
* not_visible — the page does not meet the visibility threshold, so it is not prioritized.


In [9]:
import os
import sys
import subprocess
import pandas as pd
import numpy as np

# Setup
IN_COLAB = "google.colab" in sys.modules
REPO_URL = "https://github.com/flyrank-bih/flyrank-ml-internship-starter"
REPO_DIR = "flyrank-ml-internship-starter"

if IN_COLAB:
    if not os.path.isdir(REPO_DIR):
        subprocess.run(
            ["git", "clone", "--depth", "1", REPO_URL, REPO_DIR],
            check=True
        )
    os.chdir(REPO_DIR)
    subprocess.run(
        [sys.executable, "-m", "pip", "install", "-q", "-r", "requirements.txt"],
        check=True
    )
elif os.path.basename(os.getcwd()) == "notebooks":
    os.chdir("..")

# Load starter data
df = pd.read_csv("data/raw/content_refresh_anonymized.csv")

# Outcome used ONLY to check whether the signals are useful.
# It is NOT used in the final baseline score.
df["is_declining_label"] = (
    df["trend_direction"].str.lower().eq("down").astype(int)
)

# -------------------------
# Signal 1: Staleness
# -------------------------
df["stale_bucket"] = np.where(
    df["days_since_last_update"] >= 180,
    "stale_180_plus",
    "recent_under_180"
)

stale_table = (
    df.groupby("stale_bucket")
      .agg(
          n=("is_declining_label", "size"),
          declining_rate=("is_declining_label", "mean")
      )
      .reset_index()
)

print("SIGNAL 1 — STALENESS")
display(stale_table)

# -------------------------
# Signal 2: Search visibility
# -------------------------
df["visible_bucket"] = np.where(
    df["impressions_90d"] >= 500,
    "visible_500_plus",
    "lower_than_500"
)

visible_table = (
    df.groupby("visible_bucket")
      .agg(
          n=("is_declining_label", "size"),
          declining_rate=("is_declining_label", "mean")
      )
      .reset_index()
)

print("SIGNAL 2 — SEARCH VISIBILITY")
display(visible_table)

# -------------------------
# Verdicts
# -------------------------
stale_rates = stale_table.set_index("stale_bucket")["declining_rate"]
visible_rates = visible_table.set_index("visible_bucket")["declining_rate"]

if stale_rates["stale_180_plus"] > stale_rates["recent_under_180"]:
    stale_verdict = "CONFIRMED"
elif stale_rates["stale_180_plus"] < stale_rates["recent_under_180"]:
    stale_verdict = "OPPOSITE"
else:
    stale_verdict = "MIXED"

if visible_rates["visible_500_plus"] > visible_rates["lower_than_500"]:
    visible_verdict = "CONFIRMED"
elif visible_rates["visible_500_plus"] < visible_rates["lower_than_500"]:
    visible_verdict = "OPPOSITE"
else:
    visible_verdict = "MIXED"

print("\nStaleness verdict:", stale_verdict)
print("Visibility verdict:", visible_verdict)


SIGNAL 1 — STALENESS


,stale_bucket,n,declining_rate
0,recent_under_180,29826,0.542480
1,stale_180_plus,174,0.471264


SIGNAL 2 — SEARCH VISIBILITY


,visible_bucket,n,declining_rate
0,lower_than_500,13274,0.474687
1,visible_500_plus,16726,0.595540



Staleness verdict: OPPOSITE
Visibility verdict: CONFIRMED


## 2. Build the ranked queue (writes the CSV)

*Code the score, rank everything, write work/outputs/baseline_action_score.csv.*

In [10]:
# Build the transparent baseline score from the confirmed signal.

# Higher impressions = higher priority among pages with meaningful visibility.
df["score"] = np.where(
    df["impressions_90d"] >= 500,
    df["impressions_90d"],
    0
)

# Reason code
df["reason_code"] = np.where(
    df["score"] > 0,
    "visible",
    "not_visible"
)

# Action label
df["action"] = np.where(
    df["score"] > 0,
    "REVIEW",
    "NO_ACTION"
)

# Rank highest-scoring pages first.
df["rank"] = (
    df["score"]
    .rank(method="first", ascending=False)
    .astype(int)
)

baseline_queue = (
    df[
        [
            "rank",
            "score",
            "reason_code",
            "action"
        ]
    ]
    .sort_values("rank")
    .reset_index(drop=True)
)

# Write the required CSV.
os.makedirs("work/outputs", exist_ok=True)

output_path = "work/outputs/baseline_action_score.csv"
baseline_queue.to_csv(output_path, index=False)

print("Rows written:", len(baseline_queue))
print("Output:", output_path)

display(baseline_queue.head(10))

Rows written: 30000
Output: work/outputs/baseline_action_score.csv


,rank,score,reason_code,action
0,1,517715,visible,REVIEW
1,2,517109,visible,REVIEW
2,3,509252,visible,REVIEW
3,4,497727,visible,REVIEW
4,5,463103,visible,REVIEW
5,6,443434,visible,REVIEW
6,7,416180,visible,REVIEW
7,8,347399,visible,REVIEW
8,9,345111,visible,REVIEW
9,10,312694,visible,REVIEW


## 3. Top-20 review

*For each of the top 20: action, reason code, confidence note, and what would make it wrong.*

In [11]:
# Review the top 20 ranked pages.

top20 = baseline_queue.head(20).copy()

top20["confidence_note"] = np.where(
    top20["action"] == "REVIEW",
    "Higher search visibility supports review priority.",
    "Lower search visibility means this page is not prioritized."
)

top20["what_would_make_it_wrong"] = np.where(
    top20["action"] == "REVIEW",
    "The page may not need a refresh despite having high visibility.",
    "The page may actually need review even with lower visibility."
)

top20_review = top20[
    [
        "rank",
        "action",
        "reason_code",
        "confidence_note",
        "what_would_make_it_wrong"
    ]
]

display(top20_review)

,rank,action,reason_code,confidence_note,what_would_make_it_wrong
0,1,REVIEW,visible,Higher search visibility supports review prior...,The page may not need a refresh despite having...
1,2,REVIEW,visible,Higher search visibility supports review prior...,The page may not need a refresh despite having...
2,3,REVIEW,visible,Higher search visibility supports review prior...,The page may not need a refresh despite having...
3,4,REVIEW,visible,Higher search visibility supports review prior...,The page may not need a refresh despite having...
4,5,REVIEW,visible,Higher search visibility supports review prior...,The page may not need a refresh despite having...
5,6,REVIEW,visible,Higher search visibility supports review prior...,The page may not need a refresh despite having...
6,7,REVIEW,visible,Higher search visibility supports review prior...,The page may not need a refresh despite having...
7,8,REVIEW,visible,Higher search visibility supports review prior...,The page may not need a refresh despite having...
8,9,REVIEW,visible,Higher search visibility supports review prior...,The page may not need a refresh despite having...
9,10,REVIEW,visible,Higher search visibility supports review prior...,The page may not need a refresh despite having...


## 4. Weak picks + leakage check

*Which picks look wrong and why? Confirm no product flags or future windows leaked in.*

Weak picks: The weakest REVIEW picks are the lower-ranked pages within the REVIEW group. They can be wrong because high search visibility does not guarantee that a page actually needs a content refresh.

Leakage check: The final baseline score uses only impressions_90d and the 500 visibility threshold. I did not use product flags, future-window fields, or the future decline label in the final score.


In [12]:
# This cell is for CODE (numbers, a query, a check).
# Write your text answer in the cell ABOVE this one — typing sentences here breaks Run All.
# Weak picks + leakage check

weak_picks = baseline_queue[
    baseline_queue["action"] == "REVIEW"
].tail(5)

print("WEAK PICKS")
display(weak_picks)

print("\nWhy these can be weak:")
print("They are prioritized only because of search visibility.")
print("High visibility does not guarantee that a page actually needs a content refresh.")

print("\nLEAKAGE CHECK")
print("Final score input: impressions_90d")
print("Product flags used in final score: NO")
print("Future-window or future-label fields used in final score: NO")

WEAK PICKS


,rank,score,reason_code,action
16721,16722,500,visible,REVIEW
16722,16723,500,visible,REVIEW
16723,16724,500,visible,REVIEW
16724,16725,500,visible,REVIEW
16725,16726,500,visible,REVIEW



Why these can be weak:
They are prioritized only because of search visibility.
High visibility does not guarantee that a page actually needs a content refresh.

LEAKAGE CHECK
Final score input: impressions_90d
Product flags used in final score: NO
Future-window or future-label fields used in final score: NO


## Self-check

Before you submit, confirm each line honestly:

- [ ] Every section above is filled — markdown thinking AND the code that backs it
- [ ] The notebook runs top to bottom with no errors (Runtime → Run all)
- [ ] No client names, URLs, or private queries anywhere
- [ ] My claims use careful words: observed, measured, directional, decision-support
- [ ] Committed to my repo under `work/notebooks/` — then submit your repo URL on the card. Done.